In [1]:
from src.data_ingestion.data_loader import IngestionFactory,DataLoader
from src.data_ingestion.data_validator import ValidationFactory,DataValidator
from src.data_ingestion.data_preprocessor import PreprocessingFactory
from src.features.customer_features import CustomerFeatureExtractor
from src.features.product_features import ProductFeatureExtractor
from src.features.training_data_builder import TrainingDataBuilder
from src.training.data_splitter import TemporalDataSplitter
from src.models.baseline_models import PopularityRecommender,PersonalFrequencyRecommender
from src.evaluation.metrics import RankingMetrics
from src.models.lightgbm_ranker import LightGBMRanker
from collections import Counter

In [2]:
#Load CSV
ingestion = IngestionFactory.create(
    source_type="csv",
    file_path = 'data/raw/data_raw.csv',
    date_columns=["order_date", "first_order_date", "last_order_date"]
)

loader = DataLoader(ingestion)
raw = loader.load()
df = raw.transactions

In [3]:
#Validation - only proceed when is valid = True
validator = DataValidator(
    rules=ValidationFactory.default_rules(),   # <-- Using ValidationFactory 
    strict_mode=False                          # False = allow WARNING, fail only ERROR/CRITICAL
)
report = validator.validate(df)
report.is_valid

True

In [4]:
#Preprocessing
preprocessor = PreprocessingFactory.create(
    method="sequence",
    min_orders=2
)
prepared = preprocessor.transform(df)


In [5]:
#Customer Feature Extration
cust_ext = CustomerFeatureExtractor()
customer_profiles = cust_ext.extract(prepared)

In [6]:
#Producgt Feature Extration
prod_ext = ProductFeatureExtractor()
product_features = prod_ext.extract(prepared)

In [7]:
#Build Training Data
builder = TrainingDataBuilder(negative_ratio=5)
training_data = builder.build(
prepared_data=prepared,
product_features=product_features,
customer_profiles=customer_profiles
)

In [8]:
#Train Test Split
#Issue:CustomerID in train set might not in test set
splitter = TemporalDataSplitter(test_ratio=0.2)
split = splitter.split(training_data)

In [9]:
train_df = split.train_df
test_df = split.test_df
feature_names = split.feature_names


In [10]:
#base model and baseline creation
pop_model = PopularityRecommender().fit(train_df, feature_names)
pf_model = PersonalFrequencyRecommender(smoothing=0.3).fit(train_df, feature_names)


In [11]:
#
test_scores_pop = pop_model.predict_df(test_df)
test_scores_pf = pf_model.predict_df(test_df)

In [12]:
evaluator = RankingMetrics(k_values=[1, 3, 5])

result_pop = evaluator.evaluate(pop_model, test_df, feature_names)
print(result_pop.metrics)

{'ndcg@1': 0.3308155165791832, 'ndcg@3': 0.49428661529520884, 'ndcg@5': 0.5654453639166567, 'hit_rate@1': 0.3308155165791832, 'hit_rate@3': 0.7383177570093458, 'hit_rate@5': 0.9014210728459864, 'precision@1': 0.3308155165791832, 'precision@3': 0.25523833909443944, 'precision@5': 0.20028165407758292, 'mrr': 0.5567016663793635}


In [13]:
#LGBM Ranker


# Create the model
lgbm_model = LightGBMRanker(
    num_leaves=31,
    learning_rate=0.05,
    num_boost_round=300,
    early_stopping_rounds=30
)

# Fit the model (train_df must contain label, customer_id, order_idx)
lgbm_model.fit(
    train_df=train_df,
    feature_names=feature_names
)


Training until validation scores don't improve for 30 rounds
[100]	train's ndcg@1: 0.924642	train's ndcg@3: 0.92055	train's ndcg@5: 0.93387	train's ndcg@10: 0.946182
[200]	train's ndcg@1: 0.93446	train's ndcg@3: 0.927027	train's ndcg@5: 0.939636	train's ndcg@10: 0.951475
[300]	train's ndcg@1: 0.939609	train's ndcg@3: 0.93104	train's ndcg@5: 0.943509	train's ndcg@10: 0.954741
